# KISS ICP Validation

At a chosen `TIME` this notebook:
1. Reads GPS from the rosbag and converts to Dutch RD (EPSG:28992).
2. Loads KISS ICP poses saved by `run_kiss_icp.py`.
3. Estimates bike heading from GPS displacement around `TIME`.
4. Builds a sample strip from `X_BACK` to `X_FRONT` every `DX` metres.
5. Queries AHN4 DTM once for the full strip.
6. Extracts the KISS ICP z-profile along the same strip.
7. Plots and saves both profiles.

## Settings

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import os, re


RESULTS_DIR = Path(r"C:\Users\Leons\lidar-ground-segmentation2\1. Validation Methods\KISS ICP\KISS ICP results\2026_05_22 10_50_00")
TIME = 1779439917.972930459
poses      = np.load(RESULTS_DIR / "poses.npy")       # (N, 4, 4)
timestamps = np.load(RESULTS_DIR / "timestamps.npy")   # (N,)

print(f"Loaded {len(poses)} poses from {RESULTS_DIR}/")

# =============================================================================
# PLOT
# =============================================================================

trajectory = poses[:, :3, 3]   # (N, 3)  x, y, z
x, y, z    = trajectory[:, 0], trajectory[:, 1], trajectory[:, 2]
t          = timestamps

In [ ]:
from matplotlib.pyplot import stem


ref_idx = np.argmin(np.abs(t - TIME)) #argmin returns the index of the minimum value in the array, which corresponds to the closest timestamp to TIME
dt = abs(t[ref_idx] - TIME)
print(f"Using TIME = {TIME}, closest GPS message Δt = {dt:.3f}s " f"(index {ref_idx})")


s = np.sqrt(np.diff(x)**2 + np.diff(y)**2)
cum_s = np.concatenate([[0], np.cumsum(s)])
cum_s_from_s0 = cum_s - cum_s[ref_idx]
idx_25m = np.argmax(cum_s_from_s0 >= 20.0)
idx_5m = np.argmin(cum_s_from_s0 <= -5.0)

s_rel = cum_s_from_s0[idx_5m : idx_25m + 1]
z_rel = z[idx_5m : idx_25m + 1] - z[ref_idx]
t_rel = t[idx_5m : idx_25m + 1]

def parse_bin_metadata(bin_path: str) -> dict:
    """
    Extract date_folder, time_folder, and unix timestamp from a ground .bin filename.
    Expected tail: ..._YYYY_MM_DD_HH_MM_SS_<unix_ms>.bin
    """
    stem = os.path.splitext(os.path.basename(bin_path))[0]
    pattern = r"(\d{4}_\d{2}_\d{2})[ _](\d{2}_\d{2}_\d{2})"
    m = re.search(pattern, stem)
    if not m:
        raise ValueError(f"Cannot parse metadata from filename: '{stem}'")
    return {
        "date_folder": m.group(1),              # "2026_05_01"
        "time_folder": m.group(2),              # "14_40_00"
    }
meta        = parse_bin_metadata(RESULTS_DIR)
date_folder = meta["date_folder"]   # e.g. "2026_05_01"
time_folder = meta["time_folder"]   # e.g. "14_40_00"


SAVE_DIR = os.path.join(r"D:\Validation_results", date_folder, time_folder)
os.makedirs(SAVE_DIR, exist_ok=True)

method = "EKF_curvature_validation"

np.savez_compressed(
    os.path.join(SAVE_DIR, f"{method}_{TIME}.npz"),
    x      = None,
    y      = None,
    z      = z_rel,
    s      = s_rel,
    t      = [TIME],
    method = np.array([method]),
)
fpath = os.path.join(SAVE_DIR, f"{method}_{TIME}.npz")
print(f"Saved to {fpath}")
print(f"  Core   : x, y, z, s, t, method")
print(f"  Extras : kappa_terrain_spline, kappa_terrain_pitch, kappa_path")